# BCTC Annual PDF Manager

Notebook điều phối ingestion semi-structured (PDF-first) cho BCTC năm.

Luồng gồm 2 stage:
- Stage 1: crawl danh sách công bố + download PDF + ghi metadata
- Stage 2: parse text cơ bản từ PDF (không OCR), gắn cờ `needs_ocr`

**Mặc định:** `hnx_verify_ssl=False` và `strict_bctc_annual_keyword_filter=False` — crawl `www.hnx.vn` và giữ mọi PDF để xem dữ liệu. Khi triển khai thật: bật `hnx_verify_ssl=True` (hoặc `HNX_SSL_VERIFY=1`) và có thể bật `strict_bctc_annual_keyword_filter=True` để chỉ BCTC năm.


In [1]:
import os
import sys
import warnings
import threading
from pathlib import Path

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

# SSL www.hnx.vn: mac dinh code dung hnx_verify_ssl=False. Muon bat verify TLS:
#   trong .env dat HNX_SSL_VERIFY=1  hoac  SemiStructuredIngestionConfig(hnx_verify_ssl=True)

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

_orig_hook = threading.excepthook

def _quiet_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        return
    _orig_hook(args)

threading.excepthook = _quiet_hook
warnings.filterwarnings("ignore")

root = Path.cwd()
if not (root / "ingestion").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print("[OK] UTF-8 guard + sys.path ready")

[OK] UTF-8 guard + sys.path ready


In [2]:
import os
os.environ["HNX_CRAWL_MAX_LIST_PAGES"] = "1"

In [3]:
import os

from ingestion.common import configure_logging, load_dotenv_from_project_root
from ingestion.semi_structure_data import (
    SemiStructuredIngestionConfig,
    run_bctc_annual_pipeline,
)

configure_logging()
load_dotenv_from_project_root()

if os.environ.get("HNX_SSL_VERIFY", "").strip().lower() in ("0", "false", "no"):
    print("[INFO] .env: HNX_SSL_VERIFY — tat verify SSL cho www.hnx.vn")

cfg = SemiStructuredIngestionConfig(
    sources=["hnx"],
    incremental_window_days=30,
    rate_limit_rpm=20,
    min_pdf_bytes=20_000,
    min_text_chars=500,
    # Mac dinh trong config: hnx_verify_ssl=False, strict_bctc_annual_keyword_filter=False
    # de crawl duoc tren Windows va xem du lieu. Production: bat True + HNX_SSL_VERIFY=1.
)

print(f"run_date: {cfg.run_date}")
print(f"data_lake_root: {cfg.data_lake_root}")

2026-05-13 15:42:27 [INFO] Đã nạp biến môi trường từ D:\WorkSpace\Đồ Án 2\stock-pipeline\.env


run_date: 2026-05-13
data_lake_root: D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw


In [4]:
# # Debug crawl: xem danh sách PDF HNX
# from ingestion.semi_structure_data.providers.hnx_disclosure_provider import (
#     fetch_hnx_annual_bctc_documents,
#  )

# docs = fetch_hnx_annual_bctc_documents(cfg)
# print(f"HNX docs count: {len(docs)}")
# docs[:5]

In [5]:
# Download-only
out_download = run_bctc_annual_pipeline(
    cfg,
    include_download=True,
    include_parse=False,
)
out_download

2026-05-13 15:42:27 [INFO] hnx_verify_ssl=False (mac dinh dev) — tat SSL verify cho www.hnx.vn; dat hnx_verify_ssl=True hoac HNX_SSL_VERIFY=1 khi can an toan
2026-05-13 15:42:36 [INFO] HNX attachment accepted: CAN | 1.CAN_2026.5.13_473d480_Vi_GiaiTrinhLienQuanBCTC_Q1_2026_signed.pdf
2026-05-13 15:42:36 [INFO] HNX attachment accepted: CAN | 2.CAN_2026.5.13_5e82e5c_Vi_BaoCaoTaiChinhCtyMe_Q1_2026_signed.pdf
2026-05-13 15:42:36 [INFO] HNX attachment accepted: CAN | 3.CAN_2026.5.13_cc5821c_En_ExplainationRelatingToFSs_Q1_2026_signed.pdf
2026-05-13 15:42:36 [INFO] HNX attachment accepted: CAN | 4.CAN_2026.5.13_d08e238_En_FinancialStatementsSeparate_Q1_2026_signed.pdf
2026-05-13 15:42:39 [INFO] HNX attachment accepted: VIC124005 | 1.VIC124005_2026.5.13_35e6fe8_VI_BaoCaoTaiChinh_Q1_2026.pdf
2026-05-13 15:42:39 [INFO] HNX attachment accepted: VIC124005 | 2.VIC124005_2026.5.13_7f99739_EN_FinancialStatements_Q1_2026.pdf
2026-05-13 15:42:42 [INFO] HNX attachment accepted: VC6 | 1.VC6_2026.5.13_252

KeyboardInterrupt: 

In [ ]:
# # Parse-only
# out_parse = run_bctc_annual_pipeline(
#     cfg, 
#     include_download=False,
#     include_parse=True,
# )
# out_parse

In [ ]:
# # Full pipeline
# out_full = run_bctc_annual_pipeline(
#     cfg,
#     include_download=True,
#     include_parse=True,
# )
# out_full